# Resume Screening using NLP

In [ ]:
# --- Install dependencies (only once)
!pip install streamlit sentence-transformers PyPDF2 pandas scikit-learn markovify -q


# import Libraries

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import PyPDF2
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Setup Paths

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Path to your data folder
data_path = "/content/drive/MyDrive/data"
resume_path = "/content/drive/MyDrive/MLKholoudCV2Copy.pdf"

job_file = os.path.join(data_path, "data job posts.csv")


# Load Job Data


In [ ]:
print(" Loading job dataset...")
jobs = pd.read_csv(job_file, encoding='utf-8', on_bad_lines='skip')

print(f" Loaded {len(jobs)} job postings.")
print("Available columns:", list(jobs.columns))

# Auto-detect job description column
job_col = next((c for c in jobs.columns if "description" in c.lower() or "responsibil" in c.lower()), None)
if not job_col:
    raise ValueError(" No job description column found.")
print(f" Using job description column: {job_col}")

#Clean Text Function

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9.,!? ]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


# Read Resume (PDF)

In [ ]:
def read_pdf_text(pdf_path):
    """Extract text from PDF and clean it."""
    reader = PyPDF2.PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() or ""
    return clean_text(text)

print(" Reading resume...")
resume_text = read_pdf_text(resume_path)
print(" Resume loaded! Length:", len(resume_text), "characters")

#Load  Transforemer Model

In [ ]:
print(" Loading SentenceTransformer model...")
model = SentenceTransformer('all-mpnet-base-v2')  # stronger than MiniLM
print(" Model loaded successfully!")

# Preprocess Job Descriptions

In [ ]:
print(" Cleaning job descriptions...")
job_texts = jobs[job_col].astype(str).apply(clean_text).tolist()

# Encode Resume

In [ ]:
print(" Encoding job descriptions...")
job_embs = model.encode(job_texts, show_progress_bar=True, batch_size=32)
print(" Encoding resume...")
resume_emb = model.encode([resume_text])

# Compute Similarity

In [ ]:
print(" Calculating similarity scores...")
sims = cosine_similarity(resume_emb, job_embs)[0]
jobs["match_score"] = sims

# Show Top Matches

In [ ]:
top_jobs = jobs.sort_values("match_score", ascending=False).head(10)

print("\n Top Matching Jobs for Your Resume:\n")
for i, row in top_jobs.iterrows():
    level = (
        "Senior" if row["match_score"] > 0.75
        else "Junior" if row["match_score"] > 0.5
        else "Entry"
    )
    title = row.get("Title", row.get("Job Title", "Job"))
    print(f" {title}")
    print(f"   Match Score: {row['match_score']*100:.2f}%  |  Level: {level}")
    print(f"   Description: {str(row[job_col])[:250]}...")
    print("-" * 100)